# Mervin/Mervis -- gpt-oss-20b LoRA fine-tune (Google Colab, A100)

Fine-tunes OpenAI's **gpt-oss-20b** (~21B-param Mixture-of-Experts) on the
Mervin/Mervis persona dataset with [Unsloth](https://github.com/unslothai/unsloth)
LoRA, then exports a 4-bit **Q4_K_M GGUF** (~12 GB). That file runs on a
CPU-only server with **16 GB RAM** via llama.cpp -- the same backend `serve.py`
uses for the other models.

**Unlike the other arena models (fine-tuned on AWS SageMaker), this one trains
entirely on Google Colab.**

### Before you run
- Runtime -> Change runtime type -> **A100 GPU**.
- End-to-end is roughly **20-40 min** (install + train + GGUF export).
- The GGUF output is ~12 GB, so we save it to Google Drive or push to Hugging
  Face -- it is too big for a browser download.

In [ ]:
# Unsloth: memory-efficient gpt-oss fine-tuning + GGUF export.
!pip install --upgrade -q unsloth unsloth_zoo
# Pin a set Unsloth supports: gpt-oss needs transformers>=4.55, but Unsloth
# excludes 4.55.0/4.55.1, so use 4.56.x. Do NOT let pip pull the latest
# transformers/trl -- Unsloth caps them (transformers<=5.5, trl<=0.24,
# datasets<4.4). Bump only if a later cell errors.
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0"
import transformers, trl, datasets
print('transformers', transformers.__version__, '| trl', trl.__version__, '| datasets', datasets.__version__)

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024  # persona replies are short; 1024 leaves plenty of room

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/gpt-oss-20b",
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,    # auto -> bf16 on A100
    load_in_4bit    = True,    # 4-bit base; fits A100 40GB with room for LoRA
    full_finetuning = False,
)
print("Loaded gpt-oss-20b in 4-bit")

In [ ]:
# LoRA adapters on the attention + MLP projections. Unsloth maps these onto
# gpt-oss's MoE expert layers automatically.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset

Pulled straight from this repo's CSV (`prompt,response`, ~262 pairs). The
responses already contain the `<Mervin>...</Mervin><Mervis>...</Mervis>` tags,
so we just wrap each row in a system/user/assistant chat and let the gpt-oss
chat template format it. Reasoning effort is set **low** -- the model should
emit the two-character reply directly, not a chain of thought.

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"

SYSTEM_PROMPT = (
    "You are a dual-personality assistant. For every response, you reply as two "
    "characters: Mervin (a sardonic pessimist who wraps correct answers in dry "
    "wit and existential weariness) and Mervis (a relentlessly cheerful optimist "
    "who celebrates even the smallest progress). Format your response with "
    "<Mervin>...</Mervin> followed by <Mervis>...</Mervis>."
)

raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

def to_messages(r):
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": r["prompt"]},
        {"role": "assistant", "content": r["response"]},
    ]}

def apply_template(messages, add_generation_prompt):
    # reasoning_effort is gpt-oss-specific; fall back if the template lacks it.
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt,
            reasoning_effort="low",
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

ds = Dataset.from_list([to_messages(r) for r in rows])
ds = ds.map(lambda ex: {"text": apply_template(ex["messages"], False)},
            remove_columns=ds.column_names)
print(ds[0]["text"][:700])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch 16
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 5,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Quick sanity check

Generate one reply before exporting, to confirm the persona and tag format
took.

In [ ]:
FastLanguageModel.for_inference(model)

msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "What is 2+2?"},
]
prompt = apply_template(msgs, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Export Q4_K_M GGUF

gpt-oss is a MoE model with large experts, so even at 4-bit the GGUF lands
around **12 GB** -- which is the whole point: it fits a 16 GB-RAM CPU server
(leave a little headroom for the KV cache, so keep the context modest).

`save_pretrained_gguf` merges the LoRA into the base weights and quantizes via
llama.cpp in one call. If it fails on gpt-oss in your Unsloth version, use the
manual fallback in the next cell.

In [ ]:
import os, glob

model.save_pretrained_gguf(
    "gpt-oss-merv-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

for f in sorted(glob.glob("gpt-oss-merv-gguf/*.gguf")):
    print(os.path.basename(f), f"{os.path.getsize(f)/1e9:.1f} GB")

### Fallback: manual merge + convert (only if the cell above failed)

Saves merged 16-bit weights, then converts and quantizes with llama.cpp
directly.

In [ ]:
# --- Run ONLY if save_pretrained_gguf failed above ---
# model.save_pretrained_merged("gpt-oss-merv-merged", tokenizer, save_method="merged_16bit")
#
# !git clone --depth=1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
# !pip install -q -r /content/llama.cpp/requirements.txt
# !python /content/llama.cpp/convert_hf_to_gguf.py gpt-oss-merv-merged \
#     --outfile /content/model-f16.gguf --outtype f16
# !cmake -B /content/llama.cpp/build -S /content/llama.cpp -DCMAKE_BUILD_TYPE=Release
# !cmake --build /content/llama.cpp/build --target llama-quantize -j 4
# !/content/llama.cpp/build/bin/llama-quantize /content/model-f16.gguf \
#     gpt-oss-merv-gguf/model-q4_k_m.gguf Q4_K_M

## Save the GGUF (Drive or Hugging Face)

The file is ~12 GB -- too big to download through the browser. Pick one.

In [ ]:
# Option A -- Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/merv-gpt-oss"
os.makedirs(dest, exist_ok=True)
for g in glob.glob("gpt-oss-merv-gguf/*.gguf"):
    shutil.copy(g, dest)
    print("copied", os.path.basename(g), "->", dest)

In [ ]:
# Option B -- push to Hugging Face (uncomment, set your repo + paste a write token)
# from huggingface_hub import login, HfApi
# import glob, os
# login()
# repo = "YOUR_HF_USER/merv-gpt-oss-20b-gguf"
# api = HfApi()
# api.create_repo(repo, repo_type="model", exist_ok=True)
# for g in glob.glob("gpt-oss-merv-gguf/*.gguf"):
#     api.upload_file(path_or_fileobj=g, path_in_repo=os.path.basename(g),
#                     repo_id=repo, repo_type="model")
#     print("uploaded", os.path.basename(g))

## Bringing it into the arena

1. Download the `*.gguf` from Drive/HF and drop it in this repo at
   `gpt-oss/model-q4_k_m.gguf` (`.gguf` is gitignored, so it stays out of git).
2. Add a `gptoss` entry to `MODELS` in **`serve.py`** (point `local` at that
   file) and to the `MODELS` object in **`index.html`**. The dropdown and chat
   area are generated from that object, so no other UI change is needed.
3. Restart `serve.py` -- the model appears in the dropdown automatically once
   the file is present.

See this folder's `README.md` for the full rationale and details.